# Sitting Posture Evaluation — Video Notebook

This notebook processes a **recorded video file** (not a live webcam) using YOLO person detection and MediaPipe Pose. It saves an annotated video and a CSV file of frame-by-frame pose features.

Run the cells from top to bottom. If you see a MediaPipe import/API error, run the installation cell, restart the kernel, and then run the notebook again.

## 1. Install / repair dependencies

Run this cell if the required packages are not installed or if `mediapipe` does not have `solutions`. After installing, use **Kernel → Restart Kernel** in VS Code, then run the notebook from the top.

MediaPipe `0.10.21` supports the `mp.solutions.pose` API used below.

In [11]:
%pip install opencv-python mediapipe==0.10.21 ultralytics pandas numpy

Note: you may need to restart the kernel to use updated packages.


## 2. Imports and environment check

In [12]:
import os
import math
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import mediapipe as mp
from ultralytics import YOLO

print('MediaPipe version:', getattr(mp, '__version__', 'unknown'))
print('MediaPipe loaded from:', mp.__file__)
print('Has mp.solutions:', hasattr(mp, 'solutions'))

if not hasattr(mp, 'solutions'):
    raise RuntimeError(
        'This MediaPipe import does not expose mp.solutions. Check that the notebook is using your project .venv, '
        'run the installation cell, restart the kernel, and make sure there is no local file/folder named mediapipe.'
    )

mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils
print('Imports successful.')

MediaPipe version: 0.10.21
MediaPipe loaded from: c:\Users\Owner\Downloads\sitting-posture-evaluation\.venv\Lib\site-packages\mediapipe\__init__.py
Has mp.solutions: True
Imports successful.


## 3. Configure the two video folders

Place the folders `sliced webcam` and `sliced phone` beside this notebook (normally in your project folder). Each should contain the six MP4 files listed below.

The batch processor will:
- process all 12 videos;
- save an annotated MP4 and a frame-level CSV for each video;
- combine every frame into one master CSV;
- create a per-video quality/features summary and a webcam-versus-phone comparison CSV.

If your folders are somewhere else, change `DATA_ROOT` in the next cell.


In [13]:
# Find the project folder automatically. This supports notebooks opened from
# either the project root or a subfolder such as "tests".
candidate_roots = [Path.cwd(), Path.cwd().parent]
DATA_ROOT = next(
    (root for root in candidate_roots
     if (root / "sliced webcam").is_dir() and (root / "sliced phone").is_dir()),
    Path.cwd(),
)

""" VIDEO_FOLDERS = {
    "webcam": DATA_ROOT / "sliced webcam",
    "phone": DATA_ROOT / "sliced phone",
} """
VIDEO_FOLDERS = {
    "webcam": DATA_ROOT / "sliced webcam",
    "phone": DATA_ROOT / "sliced phone"
}

OUTPUT_DIR = DATA_ROOT / "posture_output"
ANNOTATED_DIR = OUTPUT_DIR / "annotated_videos"
PER_VIDEO_CSV_DIR = OUTPUT_DIR / "per_video_csv"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ANNOTATED_DIR.mkdir(parents=True, exist_ok=True)
PER_VIDEO_CSV_DIR.mkdir(parents=True, exist_ok=True)

""" EXPECTED_FILES = [
    "front_lean_forward.mp4",
    "front_neutral.mp4",
    "front_slouch.mp4",
    "side_lean_forward.mp4",
    "side_normal.mp4",
    "side_slouch.mp4",
] """


EXPECTED_FILES = [
    "side_normal.mp4",
]

missing_folders = [str(folder) for folder in VIDEO_FOLDERS.values() if not folder.is_dir()]
if missing_folders:
    raise FileNotFoundError(
        "Could not find these input folders:\\n" + "\\n".join(missing_folders) +
        "\\nPlace both folders beside the notebook or update DATA_ROOT."
    )

missing_files = [
    str(folder / filename)
    for folder in VIDEO_FOLDERS.values()
    for filename in EXPECTED_FILES
    if not (folder / filename).is_file()
]
if missing_files:
    raise FileNotFoundError(
        "These expected videos were not found:\\n" + "\\n".join(missing_files)
    )

print("Data root:", DATA_ROOT.resolve())
print("Videos to process:", sum(len(EXPECTED_FILES) for _ in VIDEO_FOLDERS))
print("Output folder:", OUTPUT_DIR.resolve())


Data root: C:\Users\Owner\Downloads\sitting-posture-evaluation
Videos to process: 2
Output folder: C:\Users\Owner\Downloads\sitting-posture-evaluation\posture_output


## 4. Helper functions

The angles are geometric features, not posture classifications. This notebook records the features so you can inspect them and later define/test your neutral, slouching, and leaning-forward classes.

In [14]:
def calculate_angle(a, b, c):
    """Return angle ABC in degrees, or NaN if coordinates are invalid."""
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    c = np.asarray(c, dtype=float)
    if not (np.isfinite(a).all() and np.isfinite(b).all() and np.isfinite(c).all()):
        return float('nan')
    ba = a - b
    bc = c - b
    denom = np.linalg.norm(ba) * np.linalg.norm(bc)
    if denom == 0:
        return float('nan')
    cosine = np.clip(np.dot(ba, bc) / denom, -1.0, 1.0)
    return float(np.degrees(np.arccos(cosine)))


def get_xy(landmarks, landmark_id):
    """Return normalized x,y coordinates for a MediaPipe landmark."""
    lm = landmarks[landmark_id]
    return (lm.x, lm.y)


def visibility_of(landmarks, landmark_id):
    return float(landmarks[landmark_id].visibility)


def side_view_features(landmarks):
    """Extract side-view-friendly angles from MediaPipe Pose landmarks.

    Uses whichever side has greater combined shoulder/hip visibility.
    """
    L = mp_pose.PoseLandmark
    left_shoulder = L.LEFT_SHOULDER.value
    right_shoulder = L.RIGHT_SHOULDER.value
    left_elbow = L.LEFT_ELBOW.value
    right_elbow = L.RIGHT_ELBOW.value
    left_wrist = L.LEFT_WRIST.value
    right_wrist = L.RIGHT_WRIST.value
    left_hip = L.LEFT_HIP.value
    right_hip = L.RIGHT_HIP.value
    left_knee = L.LEFT_KNEE.value
    right_knee = L.RIGHT_KNEE.value
    left_ankle = L.LEFT_ANKLE.value
    right_ankle = L.RIGHT_ANKLE.value

    left_score = visibility_of(landmarks, left_shoulder) + visibility_of(landmarks, left_hip)
    right_score = visibility_of(landmarks, right_shoulder) + visibility_of(landmarks, right_hip)
    side = 'left' if left_score >= right_score else 'right'

    if side == 'left':
        shoulder, elbow, wrist = left_shoulder, left_elbow, left_wrist
        hip, knee, ankle = left_hip, left_knee, left_ankle
    else:
        shoulder, elbow, wrist = right_shoulder, right_elbow, right_wrist
        hip, knee, ankle = right_hip, right_knee, right_ankle

    nose = L.NOSE.value
    ear = L.LEFT_EAR.value if side == 'left' else L.RIGHT_EAR.value

    shoulder_xy = get_xy(landmarks, shoulder)
    elbow_xy = get_xy(landmarks, elbow)
    wrist_xy = get_xy(landmarks, wrist)
    hip_xy = get_xy(landmarks, hip)
    knee_xy = get_xy(landmarks, knee)
    ankle_xy = get_xy(landmarks, ankle)
    ear_xy = get_xy(landmarks, ear)
    nose_xy = get_xy(landmarks, nose)

    # Approximate torso inclination relative to vertical in the image plane.
    torso_dx = shoulder_xy[0] - hip_xy[0]
    torso_dy = hip_xy[1] - shoulder_xy[1]
    torso_tilt_from_vertical = float(np.degrees(np.arctan2(abs(torso_dx), abs(torso_dy)))) if abs(torso_dy) + abs(torso_dx) > 0 else float('nan')

    # Hip angle: shoulder-hip-knee. Shoulder angle: ear/shoulder/hip approximation.
    hip_angle = calculate_angle(shoulder_xy, hip_xy, knee_xy)
    shoulder_angle = calculate_angle(ear_xy, shoulder_xy, hip_xy)
    elbow_angle = calculate_angle(shoulder_xy, elbow_xy, wrist_xy)
    knee_angle = calculate_angle(hip_xy, knee_xy, ankle_xy)

    shoulder_vis = visibility_of(landmarks, shoulder)
    hip_vis = visibility_of(landmarks, hip)
    pose_reliable = (shoulder_vis >= 0.5 and hip_vis >= 0.5)

    return {
        'visible_side': side,
        'shoulder_angle_deg': shoulder_angle,
        'hip_angle_deg': hip_angle,
        'elbow_angle_deg': elbow_angle,
        'knee_angle_deg': knee_angle,
        'torso_tilt_from_vertical_deg': torso_tilt_from_vertical,
        'head_x': nose_xy[0],
        'head_y': nose_xy[1],
        'shoulder_x': shoulder_xy[0],
        'shoulder_y': shoulder_xy[1],
        'hip_x': hip_xy[0],
        'hip_y': hip_xy[1],
        'shoulder_visibility': shoulder_vis,
        'hip_visibility': hip_vis,
        'pose_reliable': bool(pose_reliable),
    }

print('Helper functions ready.')

Helper functions ready.


## 5. Batch-process all 12 videos

The filenames determine the view (`front` or `side`) and the posture class (`neutral`, `slouching`, or `leaning_forward`). The source folder determines whether a clip came from the webcam or phone.

This cell saves:
- `combined_frame_features.csv`: one row per frame across all videos;
- `per_video_summary.csv`: one row per video, including video properties, detection rates, and feature means/standard deviations;
- `webcam_vs_phone_comparison.csv`: paired comparisons for the six matching view/posture clips;
- a separate frame-level CSV and annotated MP4 for every input clip.

The same feature definitions are applied to front and side views, but some angles are more meaningful in side-view footage. Treat front-versus-side feature comparisons cautiously.


In [15]:
MODEL_NAME = "yolo11n.pt"
CONFIDENCE_THRESHOLD = 0.35
MIN_VISIBILITY = 0.5

# Standardize filename suffixes to the three posture classes.
POSTURE_LABELS = {
    "lean_forward": "leaning_forward",
    "neutral": "neutral",
    "normal": "neutral",
    "slouch": "slouching",
}

FEATURE_COLUMNS = [
    "shoulder_angle_deg", "hip_angle_deg", "elbow_angle_deg",
    "knee_angle_deg", "torso_tilt_from_vertical_deg",
    "head_x", "head_y", "shoulder_x", "shoulder_y", "hip_x", "hip_y",
    "shoulder_visibility", "hip_visibility",
]

model = YOLO(MODEL_NAME)
all_frame_tables = []
video_summaries = []

def process_one_video(video_path: Path, source_name: str):
    stem = video_path.stem
    parts = stem.lower().split("_", maxsplit=1)
    if len(parts) != 2 or parts[0] not in {"front", "side"} or parts[1] not in POSTURE_LABELS:
        raise ValueError(
            f"Unexpected video filename: {video_path.name}. "
            "Expected e.g. front_neutral.mp4 or side_lean_forward.mp4."
        )
    camera_view = parts[0]
    posture_label = POSTURE_LABELS[parts[1]]
    clip_id = stem

    annotated_path = ANNOTATED_DIR / f"{source_name}_{stem}_annotated.mp4"
    csv_path = PER_VIDEO_CSV_DIR / f"{source_name}_{stem}_features.csv"

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"OpenCV could not open video: {video_path}")

    fps = float(cap.get(cv2.CAP_PROP_FPS))
    if not np.isfinite(fps) or fps <= 0:
        fps = 30.0
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    reported_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration_seconds = reported_frames / fps if reported_frames > 0 else np.nan

    writer = cv2.VideoWriter(
        str(annotated_path),
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (width, height),
    )
    if not writer.isOpened():
        cap.release()
        raise RuntimeError(f"Could not create annotated video: {annotated_path}")

    records = []
    frame_index = 0
    pose_detector = mp_pose.Pose(
        static_image_mode=False,
        model_complexity=1,
        smooth_landmarks=True,
        enable_segmentation=False,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5,
    )
    blank_features = {
        "visible_side": None,
        "shoulder_angle_deg": np.nan,
        "hip_angle_deg": np.nan,
        "elbow_angle_deg": np.nan,
        "knee_angle_deg": np.nan,
        "torso_tilt_from_vertical_deg": np.nan,
        "head_x": np.nan, "head_y": np.nan,
        "shoulder_x": np.nan, "shoulder_y": np.nan,
        "hip_x": np.nan, "hip_y": np.nan,
        "shoulder_visibility": np.nan,
        "hip_visibility": np.nan,
        "pose_reliable": False,
    }

    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                break

            timestamp_seconds = frame_index / fps
            annotated = frame.copy()
            row = {
                "source": source_name,
                "camera_view": camera_view,
                "clip_id": clip_id,
                "video_file": video_path.name,
                "video_path": str(video_path.resolve()),
                "posture_label": posture_label,
                "frame_index": frame_index,
                "timestamp_seconds": timestamp_seconds,
                "person_detected": False,
                "pose_detected": False,
            }

            detections = model.predict(
                frame,
                classes=[0],
                conf=CONFIDENCE_THRESHOLD,
                device=0,
                verbose=False,
            )
            best_box = None
            best_conf = -1.0
            if detections and detections[0].boxes is not None:
                for box in detections[0].boxes:
                    conf = float(box.conf[0].cpu().item())
                    if conf > best_conf:
                        best_conf = conf
                        best_box = box

            if best_box is not None:
                row["person_detected"] = True
                x1, y1, x2, y2 = [
                    int(v) for v in best_box.xyxy[0].cpu().tolist()
                ]
                x1, y1 = max(0, x1), max(0, y1)
                x2, y2 = min(width, x2), min(height, y2)
                cv2.rectangle(annotated, (x1, y1), (x2, y2), (0, 200, 0), 2)
                cv2.putText(
                    annotated, f"Person {best_conf:.2f}",
                    (x1, max(20, y1 - 8)), cv2.FONT_HERSHEY_SIMPLEX,
                    0.6, (0, 200, 0), 2
                )

                rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                pose_result = pose_detector.process(rgb)
                if pose_result.pose_landmarks:
                    row["pose_detected"] = True
                    mp_drawing.draw_landmarks(
                        annotated, pose_result.pose_landmarks, mp_pose.POSE_CONNECTIONS
                    )
                    row.update(side_view_features(pose_result.pose_landmarks.landmark))
                else:
                    row.update(blank_features)
            else:
                row.update(blank_features)

            cv2.putText(
                annotated, f"Frame {frame_index}  Time {timestamp_seconds:.2f}s",
                (12, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2
            )
            writer.write(annotated)
            records.append(row)
            frame_index += 1

            if frame_index % 100 == 0:
                print(f"  {source_name}/{video_path.name}: {frame_index} frames processed")
    finally:
        cap.release()
        writer.release()
        pose_detector.close()

    frame_df = pd.DataFrame(records)
    frame_df.to_csv(csv_path, index=False)

    person_rate = float(frame_df["person_detected"].mean()) if len(frame_df) else np.nan
    pose_rate = float(frame_df["pose_detected"].mean()) if len(frame_df) else np.nan
    reliable_rate = float(frame_df["pose_reliable"].fillna(False).mean()) if len(frame_df) else np.nan

    summary = {
        "source": source_name,
        "camera_view": camera_view,
        "clip_id": clip_id,
        "video_file": video_path.name,
        "posture_label": posture_label,
        "width_px": width,
        "height_px": height,
        "fps": fps,
        "reported_frame_count": reported_frames,
        "processed_frame_count": len(frame_df),
        "duration_seconds": len(frame_df) / fps if fps > 0 else np.nan,
        "person_detection_rate": person_rate,
        "pose_detection_rate": pose_rate,
        "reliable_pose_rate": reliable_rate,
        "annotated_video": str(annotated_path.resolve()),
        "frame_features_csv": str(csv_path.resolve()),
    }

    # Per-video feature statistics, calculated over pose-reliable frames only.
    reliable_df = frame_df[frame_df["pose_reliable"].fillna(False)] if len(frame_df) else frame_df
    for feature in FEATURE_COLUMNS:
        if feature in reliable_df.columns:
            summary[f"{feature}_mean"] = reliable_df[feature].mean()
            summary[f"{feature}_std"] = reliable_df[feature].std()
    print(
        f"Finished {source_name}/{video_path.name}: {len(frame_df)} frames; "
        f"person={person_rate:.1%}; pose={pose_rate:.1%}; reliable={reliable_rate:.1%}"
    )
    return frame_df, summary

for source_name, folder in VIDEO_FOLDERS.items():
    print(f"\n=== Processing {source_name.upper()} videos in {folder} ===")
    for filename in EXPECTED_FILES:
        frame_df, summary = process_one_video(folder / filename, source_name)
        all_frame_tables.append(frame_df)
        video_summaries.append(summary)

combined_df = pd.concat(all_frame_tables, ignore_index=True)
summary_df = pd.DataFrame(video_summaries)

combined_csv_path = OUTPUT_DIR / "combined_frame_features.csv"
summary_csv_path = OUTPUT_DIR / "per_video_summary.csv"
combined_df.to_csv(combined_csv_path, index=False)
summary_df.to_csv(summary_csv_path, index=False)

print("\nBatch processing complete.")
print("Combined frame-level CSV:", combined_csv_path.resolve())
print("Per-video comparison/quality summary:", summary_csv_path.resolve())
display(summary_df[[
    "source", "camera_view", "posture_label", "video_file",
    "width_px", "height_px", "fps", "duration_seconds",
    "person_detection_rate", "pose_detection_rate", "reliable_pose_rate"
]])



=== Processing WEBCAM videos in c:\Users\Owner\Downloads\sitting-posture-evaluation\sliced webcam ===
  webcam/side_normal.mp4: 100 frames processed
  webcam/side_normal.mp4: 200 frames processed
  webcam/side_normal.mp4: 300 frames processed
  webcam/side_normal.mp4: 400 frames processed
  webcam/side_normal.mp4: 500 frames processed
  webcam/side_normal.mp4: 600 frames processed
  webcam/side_normal.mp4: 700 frames processed
  webcam/side_normal.mp4: 800 frames processed
  webcam/side_normal.mp4: 900 frames processed
Finished webcam/side_normal.mp4: 941 frames; person=100.0%; pose=100.0%; reliable=100.0%

=== Processing PHONE videos in c:\Users\Owner\Downloads\sitting-posture-evaluation\sliced phone ===
  phone/side_normal.mp4: 100 frames processed
  phone/side_normal.mp4: 200 frames processed
  phone/side_normal.mp4: 300 frames processed
  phone/side_normal.mp4: 400 frames processed
  phone/side_normal.mp4: 500 frames processed
  phone/side_normal.mp4: 600 frames processed
  phone/

,source,camera_view,posture_label,video_file,width_px,height_px,fps,duration_seconds,person_detection_rate,pose_detection_rate,reliable_pose_rate
0,webcam,side,neutral,side_normal.mp4,1280,720,30.031915,31.333333,1.0,1.0,1.0
1,phone,side,neutral,side_normal.mp4,1280,720,30.032680,30.566703,1.0,1.0,1.0


## 6. Compare the webcam and phone recordings

This comparison pairs videos with the same filename stem (for example, `side_normal`) and compares webcam and phone summary statistics. `phone_minus_webcam` is positive when the phone's mean feature value is higher.

The feature statistics use only frames where the pose was marked reliable. Detection rates are included so differences in feature values can be interpreted alongside tracking quality. A difference is descriptive, not proof that the recording device caused it.


In [16]:
# Build a paired webcam-versus-phone comparison table for each matching clip.
comparison_rows = []
numeric_features = [
    "shoulder_angle_deg", "hip_angle_deg", "elbow_angle_deg",
    "knee_angle_deg", "torso_tilt_from_vertical_deg",
    "person_detection_rate", "pose_detection_rate", "reliable_pose_rate",
    "fps", "duration_seconds", "width_px", "height_px",
]

for clip_id in sorted(summary_df["clip_id"].unique()):
    pair = summary_df[summary_df["clip_id"] == clip_id]
    webcam_rows = pair[pair["source"] == "webcam"]
    phone_rows = pair[pair["source"] == "phone"]
    if webcam_rows.empty or phone_rows.empty:
        continue
    webcam = webcam_rows.iloc[0]
    phone = phone_rows.iloc[0]
    row = {
        "clip_id": clip_id,
        "camera_view": webcam["camera_view"],
        "posture_label": webcam["posture_label"],
    }
    for feature in numeric_features:
        webcam_key = f"{feature}_mean" if feature not in {
            "person_detection_rate", "pose_detection_rate", "reliable_pose_rate",
            "fps", "duration_seconds", "width_px", "height_px"
        } else feature
        phone_key = f"{feature}_mean" if feature not in {
            "person_detection_rate", "pose_detection_rate", "reliable_pose_rate",
            "fps", "duration_seconds", "width_px", "height_px"
        } else feature
        if webcam_key in webcam.index and phone_key in phone.index:
            row[f"webcam_{feature}"] = webcam[webcam_key]
            row[f"phone_{feature}"] = phone[phone_key]
            if pd.api.types.is_numeric_dtype(type(webcam[webcam_key])) and pd.api.types.is_numeric_dtype(type(phone[phone_key])):
                row[f"phone_minus_webcam_{feature}"] = phone[phone_key] - webcam[webcam_key]
    comparison_rows.append(row)

comparison_df = pd.DataFrame(comparison_rows)
comparison_csv_path = OUTPUT_DIR / "webcam_vs_phone_comparison.csv"
comparison_df.to_csv(comparison_csv_path, index=False)

print("Webcam-versus-phone comparison CSV:", comparison_csv_path.resolve())
display(comparison_df)

print("\\nOutput files:")
print(" -", combined_csv_path.name, "(all frames from all 12 videos)")
print(" -", summary_csv_path.name, "(one summary row per video)")
print(" -", comparison_csv_path.name, "(paired webcam/phone differences)")
print(" - per_video_csv/ (12 separate frame-level CSVs)")
print(" - annotated_videos/ (12 annotated MP4s)")


Webcam-versus-phone comparison CSV: C:\Users\Owner\Downloads\sitting-posture-evaluation\posture_output\webcam_vs_phone_comparison.csv


,clip_id,camera_view,posture_label,webcam_shoulder_angle_deg,phone_shoulder_angle_deg,phone_minus_webcam_shoulder_angle_deg,webcam_hip_angle_deg,phone_hip_angle_deg,phone_minus_webcam_hip_angle_deg,webcam_elbow_angle_deg,...,phone_minus_webcam_fps,webcam_duration_seconds,phone_duration_seconds,phone_minus_webcam_duration_seconds,webcam_width_px,phone_width_px,phone_minus_webcam_width_px,webcam_height_px,phone_height_px,phone_minus_webcam_height_px
0,side_normal,side,neutral,149.689837,160.885777,11.19594,108.527048,171.982686,63.455637,140.04322,...,0.000765,31.333333,30.566703,-0.76663,1280,1280,0,720,720,0


\nOutput files:
 - combined_frame_features.csv (all frames from all 12 videos)
 - per_video_summary.csv (one summary row per video)
 - webcam_vs_phone_comparison.csv (paired webcam/phone differences)
 - per_video_csv/ (12 separate frame-level CSVs)
 - annotated_videos/ (12 annotated MP4s)


## Notes for your thesis

- The filename determines the view and class: `front`/`side`; `lean_forward` = `leaning_forward`; `neutral` and `normal` = `neutral`; `slouch` = `slouching`.
- `source` records whether the clip came from `webcam` or `phone`. This allows paired comparisons of the same nominal posture/view.
- `person_detection_rate`, `pose_detection_rate`, and `reliable_pose_rate` are the proportion of frames where each stage succeeded. They are useful quality indicators.
- Per-video feature means and standard deviations are calculated only from frames marked `pose_reliable`.
- `phone_minus_webcam_*` describes the difference between the matched clips' summary values. Differences may reflect framing, resolution, camera angle, lighting, subject movement, or tracking variation—not just the device.
- The current feature helper was designed primarily with side views in mind. Front-view angles may be less interpretable, so report results separately by `camera_view` and avoid pooling front and side data without validation.
- This notebook extracts features and compares recordings; it does not train a posture classifier.
- Expected folder layout:

  `project_folder/`

  `├── test.ipynb`

  `├── sliced webcam/` (six MP4 files)

  `├── sliced phone/` (six MP4 files)

  `└── posture_output/` (created automatically)


In [17]:
import sys
print(sys.executable)
print(sys.version)

c:\Users\Owner\Downloads\sitting-posture-evaluation\.venv\Scripts\python.exe
3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
